# BLEVE Peak Pressure Regression

## Overview

This portfolio notebook builds an end-to-end machine learning pipeline to predict peak overpressure from Boiling Liquid Expanding Vapour Explosion (BLEVE) simulation data. The workflow includes data cleaning, physics-informed feature engineering, model comparison, ensemble modelling, and explainable AI analysis.

The notebook is structured as follows:

1. Import packages
2. Load and explore data
3. Data preprocessing
4. Model development
5. Final model and prediction generation
6. Model interpretation
7. Conclusion

**Note:** The original dataset is not included in this public portfolio version. To run the notebook, place the required `train.csv`, `test.csv`, and `sample_prediction.csv` files in the working directory or upload them when prompted.


---
## 1. Import Packages

All required libraries are imported here before anything else runs. This includes data handling, machine learning models, evaluation metrics, and interpretation tools.

In [ ]:
# 1.1 Core data and plotting libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 1.2 Preprocessing — scaling and target transformation
from sklearn.preprocessing import StandardScaler, PowerTransformer

# 1.3 Model selection and cross-validation tools
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score

# 1.4 Regression models
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor

# 1.5 Evaluation metrics, MAPE and R2 are compulsory for this assignment
from sklearn.metrics import (
    r2_score, mean_absolute_percentage_error,
    mean_squared_error
)

# 1.6 Interpretation tools, PDP and permutation importance
from sklearn.inspection import PartialDependenceDisplay, permutation_importance

# 1.7 XGBoost, gradient boosting implementation explicitly allowed in the brief
from xgboost import XGBRegressor

# 1.8 SHAP for global and local model explanation
import shap

# Set a fixed random seed so all results are reproducible
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('All packages loaded successfully.')

---
## 2. Load and Explore Data

In [ ]:
# 2.1 Upload the dataset files from local storage
from google.colab import files

uploaded = files.upload()

# 2.2 Read each file into a dataframe
train_raw   = pd.read_csv('train.csv')
test_raw    = pd.read_csv('test.csv')
sample_pred = pd.read_csv('sample_prediction.csv')

# 2.3 Check the shape of each file to confirm they loaded correctly
print('train shape    :', train_raw.shape)
print('test shape     :', test_raw.shape)
print('sample_pred    :', sample_pred.shape)

In [ ]:
# 2.4 First glance at the training data
train_raw.head()

In [ ]:
# 2.5 Column types and null counts
train_raw.info()

In [ ]:
# 2.6 Descriptive statistics for all numeric columns
train_raw.describe()

In [ ]:
# 2.7 Check how many missing values exist in each dataset
print('=== TRAIN missing values ===')
print(train_raw.isnull().sum())
print('\n=== TEST missing values ===')
print(test_raw.isnull().sum())

In [ ]:
# 2.8 Inspect the Status column since it has inconsistent labels
print('Unique Status values in train:', train_raw['Status'].unique())
print('Unique Status values in test: ', test_raw['Status'].unique())

In [ ]:
# 2.9 Plot the target pressure distribution before and after log transformation
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(train_raw['Target Pressure (bar)'].dropna(), bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Target Pressure Distribution')
axes[0].set_xlabel('Target Pressure (bar)')
axes[0].set_ylabel('Count')

axes[1].hist(np.log1p(train_raw['Target Pressure (bar)'].dropna()), bins=60, color='coral', edgecolor='white')
axes[1].set_title('log1p(Target Pressure) Distribution')
axes[1].set_xlabel('log1p(Target Pressure)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

Looking at the data, a few things stand out that will shape the preprocessing decisions made in the next section.

The training set has 10,050 rows and 25 columns, while the test set is identical in structure but without the target column. Missing values are scattered across most columns in training, and since the target itself is missing for 6 rows, those rows get dropped entirely because training on an unknown target is not possible. All other missing values are filled in using statistics computed from the training data only to avoid any leakage into the test set.

The `Status` column is particularly messy, with the same two liquid states written in 9 different ways across the dataset. This gets standardised into just `Subcooled` and `Superheated` during preprocessing.

The target pressure distribution is heavily right-skewed, with most sensor readings below 1 bar but a small number of extreme cases reaching all the way to 9.17 bar. The log1p version on the right shows a much more symmetric shape, which helps explain why applying a PowerTransformer to the target improves performance for gradient boosting and XGBoost models.

One other thing worth noting is that `Tank Failure Pressure` has a maximum of 4882 bar while the mean sits at only 37.98 bar, meaning a small number of very high-pressure explosion scenarios are pulling the distribution up significantly. Tree-based models handle this kind of extreme value naturally without needing any special treatment.

---
## 3. Data Preprocessing

Before building any models, the data needs to be cleaned, missing values handled, and new features created. All preprocessing statistics are computed from the training set only to avoid data leakage into the test set.

### 3.1 Cleaning

The raw data is copied first so the originals stay untouched. Rows with a missing target are dropped, the index column is removed, and the Status labels are standardised.

In [ ]:
# 3.1.1 Work on copies so the raw dataframes stay unchanged
train = train_raw.copy()
test  = test_raw.copy()

# 3.1.2 Drop rows where the target pressure is missing since we cannot train on them
train = train.dropna(subset=['Target Pressure (bar)']).reset_index(drop=True)
print(f'Train after dropping missing target: {train.shape}')

# 3.1.3 Drop the Unnamed: 0 column which is just a row index carried over from the CSV
for df in [train, test]:
    if 'Unnamed: 0' in df.columns:
        df.drop(columns=['Unnamed: 0'], inplace=True)

print('Unnamed: 0 removed from both sets.')

In [ ]:
# 3.1.4 Standardise the Status column
# The same two classes appear in 9 different spellings across the dataset
# Any value starting with sub maps to Subcooled
# Any value containing sup or starting with sa maps to Superheated
def clean_status(val):
    if pd.isna(val):
        return np.nan
    v = str(val).strip().lower()
    if v.startswith('sub'):
        return 'Subcooled'
    elif 'sup' in v or v.startswith('sa'):
        return 'Superheated'
    return np.nan

train['Status'] = train['Status'].apply(clean_status)
test['Status']  = test['Status'].apply(clean_status)

print('Status values after cleaning:')
print('  train:', train['Status'].value_counts(dropna=False).to_dict())
print('  test: ', test['Status'].value_counts(dropna=False).to_dict())

In [ ]:
# 3.1.5 Fill missing values using training statistics only
# Numeric columns are filled with the median from training
# The Status column is filled with the most common class from training
# Fitting on test data would leak information so everything is computed from train

numeric_cols  = train.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols  = [c for c in numeric_cols if c != 'Target Pressure (bar)']
train_medians = train[numeric_cols].median()
status_mode   = train['Status'].mode()[0]

train[numeric_cols] = train[numeric_cols].fillna(train_medians)
test[numeric_cols]  = test[numeric_cols].fillna(train_medians)

train['Status'] = train['Status'].fillna(status_mode)
test['Status']  = test['Status'].fillna(status_mode)

print('Missing values after imputation:')
print('  train:', train.isnull().sum().sum())
print('  test: ', test.isnull().sum().sum())

### 3.2 Feature Engineering

Rather than leaving the models to figure out physical relationships from raw columns alone, I created new features that encode domain knowledge about blast wave behaviour directly. Each feature group below captures a different aspect of the physical setup.

In [ ]:
def add_engineered_features(df):
    eps = 1e-9
    d = df.copy()

    # 3.2.1 Tank geometry features
    # Total internal volume of the tank
    d['Tank Volume']             = d['Tank Width (m)'] * d['Tank Length (m)'] * d['Tank Height (m)']
    # Shape ratios to capture whether the tank is tall, wide or elongated
    d['Tank Width Length Ratio'] = d['Tank Width (m)'] / (d['Tank Length (m)'] + eps)
    d['Tank Height Width Ratio'] = d['Tank Height (m)'] / (d['Tank Width (m)'] + eps)

    # 3.2.2 Vapour and liquid condition features
    # Fraction of the tank height filled by vapour
    d['Vapour Height Ratio']    = d['Vapour Height (m)'] / (d['Tank Height (m)'] + eps)
    # Height of the liquid column
    d['Liquid Height']          = d['Tank Height (m)'] - d['Vapour Height (m)']
    # Temperature gap between vapour and liquid phases
    d['Temperature Difference'] = d['Vapour Temperature (K)'] - d['Liquid Temperature (K)']
    # How far the vapour temperature is above the liquid boiling point
    d['Superheat Gap']          = d['Vapour Temperature (K)'] - d['Liquid Boiling Temperature (K)']

    # 3.2.3 Reduced thermodynamic quantities
    # Dimensionless pressure relative to the critical pressure of the substance
    d['Reduced Pressure']    = d['Tank Failure Pressure (bar)'] / (d['Liquid Critical Pressure (bar)'] + eps)
    # Dimensionless temperature relative to the critical temperature
    d['Reduced Temperature'] = d['Vapour Temperature (K)'] / (d['Liquid Critical Temperature (K)'] + eps)

    # 3.2.4 Obstacle geometry features
    # Cross sectional area and volume of the obstacle wall
    d['Obstacle Area']         = d['Obstacle Width (m)'] * d['Obstacle Height (m)']
    d['Obstacle Volume']       = d['Obstacle Width (m)'] * d['Obstacle Height (m)'] * d['Obstacle Thickness (m)']
    # Aspect ratio of the obstacle face
    d['Obstacle Aspect Ratio'] = d['Obstacle Width (m)'] / (d['Obstacle Height (m)'] + eps)

    # 3.2.5 Sensor distance and geometry features
    x  = d['Sensor Position x']
    y  = d['Sensor Position y']
    z  = d['Sensor Position z']
    bh = d['BLEVE Height (m)']

    # Straight line distance from origin to sensor
    d['Sensor Distance']          = np.sqrt(x**2 + y**2 + z**2)
    # Distance from sensor to the BLEVE source accounting for height
    d['Sensor Distance to BLEVE'] = np.sqrt(x**2 + y**2 + (z - bh)**2)
    # Horizontal radial distance ignoring height
    d['Sensor Radial Distance']   = np.sqrt(x**2 + y**2)
    # Vertical offset between sensor and BLEVE source
    d['Sensor Height Difference'] = z - bh
    # How far the sensor sits behind or in front of the obstacle
    d['Obstacle Sensor Distance Difference'] = d['Obstacle Distance to BLEVE (m)'] - x
    # Direction angles from BLEVE source to sensor
    d['Sensor Azimuth']   = np.arctan2(y, x)
    d['Sensor Elevation'] = np.arctan2(d['Sensor Height Difference'], d['Sensor Radial Distance'] + eps)

    # 3.2.6 Pressure and energy proxy features
    # Total mechanical energy stored in the pressurised tank
    d['Pressure Volume']       = d['Tank Failure Pressure (bar)'] * d['Tank Volume']
    # Contribution of the vapour fraction to available pressure
    d['Vapour Pressure Proxy'] = d['Tank Failure Pressure (bar)'] * d['Vapour Height Ratio']
    # Thermal energy from superheat scaled by tank size
    d['Superheat Volume']      = d['Superheat Gap'] * d['Tank Volume']
    # Combined energy proxy combining pressure, volume and vapour fraction
    d['Energy Proxy']          = d['Tank Failure Pressure (bar)'] * d['Tank Volume'] * d['Vapour Height Ratio']

    # Scaled sensor distance normalised by the cube root of energy proxy
    # This is analogous to the Hopkinson-Cranz scaled distance used in blast engineering
    energy_cbrt = np.cbrt(np.abs(d['Energy Proxy']) + eps)
    d['Scaled Sensor Distance']   = d['Sensor Distance to BLEVE'] / (energy_cbrt + eps)
    d['Scaled Obstacle Distance'] = d['Obstacle Distance to BLEVE (m)'] / (energy_cbrt + eps)

    return d

# Apply to both training and test sets
train_fe = add_engineered_features(train)
test_fe  = add_engineered_features(test)

print(f'Features after engineering — train: {train_fe.shape[1]}, test: {test_fe.shape[1]}')

### 3.3 Encode Categorical Features and Prepare Feature Sets

Two feature sets are prepared because treating Sensor ID and Sensor Position Side as categories rather than numbers turned out to help on the Kaggle public leaderboard in earlier experiments.

The baseline set only dummy-codes Status. The sensor-coded set also dummy-codes Sensor ID and Sensor Position Side by first converting them to strings so pandas treats them as categories rather than numeric values.

In [ ]:
# 3.3.1 Separate the target variable from the features
TARGET    = 'Target Pressure (bar)'
y_full    = train_fe[TARGET].values
feat_cols = [c for c in train_fe.columns if c != TARGET]

X_all  = train_fe[feat_cols].copy()
X_test = test_fe[feat_cols].copy()

# Drop index column if it was carried through feature engineering
if 'Unnamed: 0' in X_all.columns:
    X_all  = X_all.drop('Unnamed: 0', axis=1)
    X_test = X_test.drop('Unnamed: 0', axis=1)

# 3.3.2 Baseline encoding, Status only dummy-coded
X_base      = pd.get_dummies(X_all,  columns=['Status'], drop_first=True)
X_test_base = pd.get_dummies(X_test, columns=['Status'], drop_first=True)
X_test_base = X_test_base.reindex(columns=X_base.columns, fill_value=0)

print('Baseline feature set shape — train:', X_base.shape, ' test:', X_test_base.shape)

In [ ]:
# 3.3.3 Sensor-coded encoding, Status plus Sensor ID and Sensor Position Side
# Converting to string first forces pandas to treat them as nominal categories
X_sc      = X_all.copy()
X_test_sc = X_test.copy()

for col in ['Sensor ID', 'Sensor Position Side']:
    X_sc[col]      = X_sc[col].astype(str)
    X_test_sc[col] = X_test_sc[col].astype(str)

X_sc      = pd.get_dummies(X_sc,      columns=['Status', 'Sensor ID', 'Sensor Position Side'], drop_first=True)
X_test_sc = pd.get_dummies(X_test_sc, columns=['Status', 'Sensor ID', 'Sensor Position Side'], drop_first=True)
X_test_sc = X_test_sc.reindex(columns=X_sc.columns, fill_value=0)

print('Sensor-coded feature set shape — train:', X_sc.shape, ' test:', X_test_sc.shape)

In [ ]:
# 3.3.4 Split into training and validation sets using an 80/20 split
# The same split indices are applied to both feature sets so all models
# are evaluated on the exact same validation rows
X_train_b, X_valid_b, y_train, y_valid = train_test_split(
    X_base, y_full, test_size=0.2, random_state=RANDOM_STATE
)
X_train_sc, X_valid_sc = X_sc.iloc[X_train_b.index], X_sc.iloc[X_valid_b.index]

print('Train size:', X_train_b.shape[0], '  Validation size:', X_valid_b.shape[0])

---
## 4. Model Development

I tested 12 different model configurations to find the best approach for predicting BLEVE peak pressure. All models are evaluated on the same 80/20 validation split so comparisons are fair. The two required metrics are MAPE and R². Since Kaggle scores on MAPE, that is the primary metric for model selection.

In [ ]:
# 4.0 Helper function to evaluate any model and store results
# Computes R2, MAPE, MSE and RMSE then appends to the results list
results = []

def evaluate_model(name, y_true, y_pred):
    r2   = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    results.append({'Model': name, 'R2': round(r2, 4),
                    'MAPE': round(mape, 4), 'MSE': round(mse, 4), 'RMSE': round(rmse, 4)})
    print(f"{name:<60}  R2={r2:.4f}  MAPE={mape:.4f}  RMSE={rmse:.4f}")

### 4.1 Model 1 - Linear Regression

Linear Regression is used as a baseline to understand how much a simple model can do on this problem. Features are scaled with StandardScaler since linear models are sensitive to feature magnitude.

In [ ]:
# 4.1.1 Scale features since linear models are sensitive to feature magnitude
scaler_lin     = StandardScaler()
X_train_scaled = scaler_lin.fit_transform(X_train_b)
X_valid_scaled = scaler_lin.transform(X_valid_b)

# 4.1.2 Fit Linear Regression and evaluate on the validation set
lr        = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_valid_scaled)

evaluate_model('Linear Regression', y_valid, y_pred_lr)

### 4.2 Model 2 - Lasso Regression

Lasso adds L1 regularisation to linear regression, which forces some feature coefficients to zero and produces a sparse model. GridSearchCV is used to find the best regularisation strength alpha across 5 folds.

In [ ]:
# 4.2.1 Search for the best alpha using 5-fold cross-validation
# Lasso with a higher alpha pushes more coefficients to zero
lasso_grid = GridSearchCV(
    Lasso(max_iter=5000),
    param_grid={'alpha': [0.0001, 0.001, 0.01, 0.1, 1.0]},
    cv=5, scoring='neg_mean_absolute_percentage_error', n_jobs=-1
)
lasso_grid.fit(X_train_scaled, y_train)

best_lasso   = lasso_grid.best_estimator_
print(f'Best Lasso alpha: {lasso_grid.best_params_["alpha"]}')

# 4.2.2 Evaluate on validation set
y_pred_lasso = best_lasso.predict(X_valid_scaled)
evaluate_model('Lasso Regression (tuned alpha)', y_valid, y_pred_lasso)

### 4.3 Model 3 - Random Forest (default settings)

Random Forest is tested first with default settings to get a quick sense of how tree-based models perform on this problem before any tuning is done.

In [ ]:
# 4.3.1 Random Forest with default hyperparameters as a starting point
rf_default        = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf_default.fit(X_train_b, y_train)
y_pred_rf_default = rf_default.predict(X_valid_b)

evaluate_model('Random Forest (default)', y_valid, y_pred_rf_default)

### 4.4 Model 4 - Random Forest with GridSearchCV Tuning

GridSearchCV is used here to search across different combinations of hyperparameters using 3-fold cross-validation. This is a more systematic way of tuning than guessing values manually.

In [ ]:
# 4.4.1 Define the hyperparameter grid to search over
rf_param_grid = {
    'n_estimators':      [200, 400],
    'max_depth':         [None, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf':  [1, 2],
}

# 4.4.2 Run 3-fold cross-validation across all parameter combinations
rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid=rf_param_grid,
    cv=3, scoring='neg_mean_absolute_percentage_error', n_jobs=-1, verbose=1
)
rf_grid.fit(X_train_b, y_train)

print('Best RF params:', rf_grid.best_params_)
rf_tuned        = rf_grid.best_estimator_
y_pred_rf_tuned = rf_tuned.predict(X_valid_b)

evaluate_model('Random Forest (GridSearchCV tuned)', y_valid, y_pred_rf_tuned)

### 4.5 Model 5 - Gradient Boosting with GridSearchCV Tuning

Gradient Boosting builds trees sequentially where each tree corrects the errors of the previous one. GridSearchCV is used to tune the learning rate, tree depth, and number of estimators.

In [ ]:
# 4.5.1 Define the hyperparameter grid for Gradient Boosting
gb_param_grid = {
    'n_estimators':  [200, 400],
    'learning_rate': [0.05, 0.1],
    'max_depth':     [4, 6],
}

# 4.5.2 Run 3-fold cross-validation
gb_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=RANDOM_STATE),
    param_grid=gb_param_grid,
    cv=3, scoring='neg_mean_absolute_percentage_error', n_jobs=-1, verbose=1
)
gb_grid.fit(X_train_b, y_train)

print('Best GB params:', gb_grid.best_params_)
gb_tuned    = gb_grid.best_estimator_
y_pred_gb   = gb_tuned.predict(X_valid_b)

evaluate_model('Gradient Boosting (tuned)', y_valid, y_pred_gb)

### 4.6 Model 6 - Random Forest with Sensor Dummy-Coded Features

This tests whether treating Sensor ID and Sensor Position Side as nominal categories rather than numeric values improves the Random Forest. Both columns are dummy-coded along with Status.

In [ ]:
# 4.6.1 Random Forest on the sensor-coded feature set
# Sensor ID and Sensor Position Side are treated as categories here
rf_sc = RandomForestRegressor(
    n_estimators=400, max_depth=None,
    min_samples_split=2, min_samples_leaf=1,
    random_state=RANDOM_STATE, n_jobs=-1
)
rf_sc.fit(X_train_sc, y_train)
y_pred_rf_sc = rf_sc.predict(X_valid_sc)

evaluate_model('Random Forest (sensor dummy-coded)', y_valid, y_pred_rf_sc)

### 4.7 Model 7 - Random Forest with PowerTransformer on Target

A Yeo-Johnson PowerTransformer is applied to the target variable before training to reduce the effect of the right skew. The transformer is fitted on the training target only to avoid leakage, and predictions are clipped to the training range before inverse transforming.

In [ ]:
# 4.7.1 Fit PowerTransformer on the training target only to avoid leakage
pt           = PowerTransformer(method='yeo-johnson')
y_train_pt   = pt.fit_transform(y_train.reshape(-1, 1)).ravel()
y_train_pt_min, y_train_pt_max = y_train_pt.min(), y_train_pt.max()

# 4.7.2 Train Random Forest on the transformed target
rf_pt = RandomForestRegressor(
    n_estimators=400, max_depth=None,
    min_samples_split=2, min_samples_leaf=1,
    random_state=RANDOM_STATE, n_jobs=-1
)
rf_pt.fit(X_train_b, y_train_pt)
y_pred_rf_pt_raw = rf_pt.predict(X_valid_b)

# 4.7.3 Clip then inverse transform to get back to original pressure scale
y_pred_rf_pt_raw = np.clip(y_pred_rf_pt_raw, y_train_pt_min, y_train_pt_max)
y_pred_rf_pt     = pt.inverse_transform(y_pred_rf_pt_raw.reshape(-1, 1)).ravel()

evaluate_model('Random Forest (PowerTransformer target)', y_valid, y_pred_rf_pt)

### 4.8 Model 8 - Random Forest with log1p Target Transformation

log1p is a simpler alternative to PowerTransformer for compressing a right-skewed target. Predictions are reversed with expm1 after inference.

In [ ]:
# 4.8.1 Train on log1p transformed target and reverse with expm1
rf_log        = RandomForestRegressor(
    n_estimators=400, max_depth=None,
    min_samples_split=2, min_samples_leaf=1,
    random_state=RANDOM_STATE, n_jobs=-1
)
rf_log.fit(X_train_b, np.log1p(y_train))
y_pred_rf_log = np.expm1(rf_log.predict(X_valid_b))

evaluate_model('Random Forest (log1p target)', y_valid, y_pred_rf_log)

### 4.9 Model 9 - Gradient Boosting with PowerTransformer on Target

This is the final tuned model used for prediction. The hyperparameters were found through a systematic search beyond GridSearchCV, tuning `max_depth`, `subsample`, `min_samples_leaf`, `n_estimators` and `learning_rate` together. This configuration improved both MAPE and R² simultaneously compared to the GridSearchCV baseline.

In [ ]:
# 4.9.1 Gradient Boosting with tuned hyperparameters and PowerTransformer target
# Parameters were found through systematic fine-tuning after GridSearchCV
gb_pt = GradientBoostingRegressor(
    n_estimators=2000, learning_rate=0.03, max_depth=7,
    subsample=0.9, min_samples_leaf=5,
    random_state=RANDOM_STATE
)
gb_pt.fit(X_train_b, y_train_pt)
y_pred_gb_pt_raw = gb_pt.predict(X_valid_b)

# 4.9.2 Clip then inverse transform
y_pred_gb_pt_raw = np.clip(y_pred_gb_pt_raw, y_train_pt_min, y_train_pt_max)
y_pred_gb_pt     = pt.inverse_transform(y_pred_gb_pt_raw.reshape(-1, 1)).ravel()
y_pred_gb_pt     = np.maximum(y_pred_gb_pt, 0)

evaluate_model('Gradient Boosting (PowerTransformer target)', y_valid, y_pred_gb_pt)

### 4.10 Model 10 - XGBoost with PowerTransformer on Target

XGBoost is explicitly listed as an allowed model in the project brief. The hyperparameters were tuned systematically, with `max_depth=6`, `subsample=0.9`, and `min_child_weight=5` providing the best balance of MAPE and R².

In [ ]:
# 4.10.1 XGBoost with tuned hyperparameters and PowerTransformer target
xgb_model = XGBRegressor(
    n_estimators=4000, learning_rate=0.01, max_depth=7,
    subsample=0.9, colsample_bytree=0.9,
    min_child_weight=5, reg_alpha=0.05, reg_lambda=1.0,
    objective='reg:squarederror',
    random_state=RANDOM_STATE, n_jobs=-1, verbosity=0
)
xgb_model.fit(X_train_b, y_train_pt,
              eval_set=[(X_valid_b, pt.transform(y_valid.reshape(-1,1)).ravel())],
              verbose=False)

# 4.10.2 Clip then inverse transform
y_pred_xgb_raw = xgb_model.predict(X_valid_b)
y_pred_xgb_raw = np.clip(y_pred_xgb_raw, y_train_pt_min, y_train_pt_max)
y_pred_xgb     = pt.inverse_transform(y_pred_xgb_raw.reshape(-1, 1)).ravel()
y_pred_xgb     = np.maximum(y_pred_xgb, 0)

evaluate_model('XGBoost (PowerTransformer target)', y_valid, y_pred_xgb)

### 4.11 Model 11 - XGBoost with Sensor Dummy-Coded Features

This tests XGBoost on the sensor-coded feature set to see whether treating Sensor ID and Sensor Position Side as categories helps. The predictions are clipped and cleaned carefully because sensor dummy coding can cause compressed predictions on the test set.

In [ ]:
# 4.11.1 XGBoost on sensor-coded features
xgb_sc = XGBRegressor(
    n_estimators=1500, learning_rate=0.01, max_depth=5,
    subsample=0.85, colsample_bytree=0.85,
    min_child_weight=3, reg_alpha=0.05, reg_lambda=1.0,
    objective='reg:squarederror',
    random_state=RANDOM_STATE, n_jobs=-1, verbosity=0
)
xgb_sc.fit(X_train_sc, y_train_pt,
           eval_set=[(X_valid_sc, pt.transform(y_valid.reshape(-1, 1)).ravel())],
           verbose=False)

# 4.11.2 Clip, inverse transform and clean any remaining invalid values
y_pred_xgb_sc_raw = xgb_sc.predict(X_valid_sc)
y_pred_xgb_sc_raw = np.clip(y_pred_xgb_sc_raw, y_train_pt_min, y_train_pt_max)
y_pred_xgb_sc     = pt.inverse_transform(y_pred_xgb_sc_raw.reshape(-1, 1)).ravel()
y_pred_xgb_sc     = np.nan_to_num(y_pred_xgb_sc, nan=0.0,
                                   posinf=np.nanmax(y_valid), neginf=0.0)
y_pred_xgb_sc     = np.maximum(y_pred_xgb_sc, 0)

print(f'NaN values: {np.isnan(y_pred_xgb_sc).sum()}  Min: {y_pred_xgb_sc.min():.4f}  Max: {y_pred_xgb_sc.max():.4f}')
evaluate_model('XGBoost (sensor dummy-coded + PowerTransformer)', y_valid, y_pred_xgb_sc)

### 4.12 Model 12 - MLPRegressor (Neural Network)

Neural networks were covered in the unit. An MLP with three hidden layers is tested here. StandardScaler is required because neural networks are sensitive to feature magnitude. The PowerTransformer is applied to the target for consistency with the other models.

In [ ]:
# 4.12.1 Scale features since MLP is sensitive to feature magnitude
scaler_mlp  = StandardScaler()
X_train_mlp = scaler_mlp.fit_transform(X_train_b)
X_valid_mlp = scaler_mlp.transform(X_valid_b)

# 4.12.2 Train MLP with three hidden layers and early stopping
mlp = MLPRegressor(
    hidden_layer_sizes=(256, 128, 64), activation='relu',
    learning_rate_init=0.001, max_iter=500,
    early_stopping=True, validation_fraction=0.1,
    random_state=RANDOM_STATE
)
mlp.fit(X_train_mlp, y_train_pt)

# 4.12.3 Clip then inverse transform
y_pred_mlp_raw = mlp.predict(X_valid_mlp)
y_pred_mlp_raw = np.clip(y_pred_mlp_raw, y_train_pt_min, y_train_pt_max)
y_pred_mlp     = pt.inverse_transform(y_pred_mlp_raw.reshape(-1, 1)).ravel()

evaluate_model('MLPRegressor (PowerTransformer target)', y_valid, y_pred_mlp)

### 4.13 Model 13 - LightGBM with PowerTransformer Target

LightGBM is a gradient boosting framework that uses histogram-based splits, making it faster and often more accurate than sklearn's GradientBoostingRegressor on tabular data. Since XGBoost is explicitly listed as an allowed model in the brief, LightGBM is a suitable alternative in the same gradient boosting family.

In [ ]:
# 4.13.1 LightGBM with PowerTransformer target
import lightgbm as lgb

pt_lgb = PowerTransformer(method='yeo-johnson')
y_train_pt_lgb = pt_lgb.fit_transform(y_train.reshape(-1,1)).ravel()

lgb_model = lgb.LGBMRegressor(
    n_estimators=8000, learning_rate=0.01,
    max_depth=-1, num_leaves=63,
    subsample=0.9, colsample_bytree=0.9,
    min_child_samples=5,
    reg_alpha=0.05, reg_lambda=1.0,
    random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1
)
lgb_model.fit(X_train_b, y_train_pt_lgb)
y_pred_lgb_raw = lgb_model.predict(X_valid_b)
y_pred_lgb_raw = np.clip(y_pred_lgb_raw,
                          y_train_pt_lgb.min(), y_train_pt_lgb.max())
y_pred_lgb = np.maximum(
    pt_lgb.inverse_transform(y_pred_lgb_raw.reshape(-1,1)).ravel(), 0
)
evaluate_model('LightGBM (PowerTransformer target)', y_valid, y_pred_lgb)

### Final Model Selection

After testing all 12 models and various ensemble combinations, the final prediction uses a weighted average of Gradient Boosting with PowerTransformer target (approximately 20% weight) and LightGBM with PowerTransformer target (approximately 80% weight). This ensemble achieved the best public Kaggle MAPE of **0.161348**.

The LightGBM model was systematically tuned using `num_leaves=63`, unlimited `max_depth`, `n_estimators=8000` and `learning_rate=0.01`, which gave the best validation MAPE of 0.1000 - the strongest individual model tested. Combined with the tuned Gradient Boosting model, the ensemble achieved validation MAPE of 0.0997 and R²=0.91.

The R² values for the PowerTransformer models appear lower than the non-transformed models because clipping before inverse transformation compresses the high-pressure tail. The GB tuned model without transformation achieves R²=0.948, confirming the model is genuinely learning the data structure. MAPE was prioritised for final selection since that is the Kaggle metric.

### 4.13 Model Comparison

All 12 models are compared below. The table is sorted by validation MAPE since that is the Kaggle evaluation criterion. R² is included as the second required metric.

In [ ]:
# 4.13.1 Build comparison table sorted by MAPE
results_df = pd.DataFrame(results).sort_values('MAPE').reset_index(drop=True)
results_df

The tuned Gradient Boosting model with PowerTransformer target achieves the lowest validation MAPE of 0.1054 and is the backbone of the final ensemble. The R² of 0.88 is strong and both metrics improved together after the systematic tuning of `max_depth`, `subsample`, and `min_samples_leaf`, which was not possible through GridSearchCV alone given the search space size.

In [ ]:
# 4.13.2 Plot MAPE and R2 side by side for all models
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh(results_df['Model'], results_df['MAPE'], color='steelblue')
axes[0].set_xlabel('Validation MAPE')
axes[0].set_title('Model Comparison — MAPE (lower is better)')
axes[0].invert_yaxis()

axes[1].barh(results_df['Model'], results_df['R2'], color='coral')
axes[1].set_xlabel('Validation R²')
axes[1].set_title('Model Comparison — R² (higher is better)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

**MAPE chart (left):** The PowerTransformer models achieve the lowest MAPE, well below the Random Forest and linear models. Linear Regression and Lasso both score above 1.0 on MAPE, confirming this is a strongly nonlinear problem.

**R² chart (right):** The PowerTransformer models show lower R² than the non-transformed models. This is not because they are worse models - it is a side effect of clipping before inverse transformation which compresses extreme predictions and reduces R². The non-transformed GB model at R²=0.948 shows the tree models are genuinely capturing the data. The tuned GB with PowerTransformer achieves R²=0.88, which is the best balance between R² and MAPE of any model tested.

**Final selection:** The weighted ensemble of tuned GB_PT and LightGBM was selected based on lowest validation MAPE, which is the Kaggle criterion. The ensemble reduces variance compared to either model alone.

### 4.14 Ensemble Methods

Combining predictions from multiple models can reduce variance and improve generalisation. Two ensemble strategies are tested here, going from simple to optimised.

In [ ]:
# 4.14.0 Collect validation predictions from the top performing models
# All predictions are on the same validation fold in original pressure units
val_preds = {
    'RF_tuned':      y_pred_rf_tuned,
    'RF_sensor':     y_pred_rf_sc,
    'RF_PT':         y_pred_rf_pt,
    'GB_PT':         y_pred_gb_pt,
    'XGB_PT':        y_pred_xgb,
    'XGB_sensor_PT': y_pred_xgb_sc,
}

print('Individual model validation MAPEs:')
for name, pred in val_preds.items():
    m = mean_absolute_percentage_error(y_valid, pred)
    print(f'  {name:<20}  MAPE={m:.4f}')

#### 4.14.1 Simple Averaging

The simplest ensemble approach is to take an equal-weight average across all selected models. No optimisation is needed and it gives a useful baseline for comparing the more involved methods.

In [ ]:
# 4.14.1 Equal weight average across all models
pred_avg = np.mean(list(val_preds.values()), axis=0)

mape_avg = mean_absolute_percentage_error(y_valid, pred_avg)
r2_avg   = r2_score(y_valid, pred_avg)
print(f'Simple Averaging — MAPE={mape_avg:.4f}  R2={r2_avg:.4f}')
evaluate_model('Ensemble — Simple Average', y_valid, pred_avg)

#### 4.14.2 Weighted Averaging

Instead of equal weights, the contribution of each model is optimised on the validation set to minimise MAPE. The optimiser uses SLSQP with bounds to prevent any single model from completely dominating the ensemble.

In [ ]:
# 4.14.2 Optimise ensemble weights on the validation set to minimise MAPE
from scipy.optimize import minimize

val_mat2 = np.column_stack([y_pred_gb_pt, y_pred_lgb])

def mape_loss(weights):
    w = np.abs(weights) / (np.abs(weights).sum() + 1e-9)
    return mean_absolute_percentage_error(y_valid, val_mat2 @ w)

res = minimize(mape_loss, [0.3, 0.7], method='SLSQP',
               bounds=[(0.05, 0.95), (0.05, 0.95)],
               constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1},
               options={'ftol': 1e-10, 'maxiter': 2000})

opt_weights = np.abs(res.x) / np.abs(res.x).sum()
pred_wavg   = val_mat2 @ opt_weights

print('Optimised weights:')
print(f'  GB_PT   weight={opt_weights[0]:.4f}')
print(f'  LGB     weight={opt_weights[1]:.4f}')
print()
evaluate_model('Ensemble — GB_PT + LightGBM', y_valid, pred_wavg)

In [ ]:
# 4.14.3 Retrain both models on the full training dataset
pt_full   = PowerTransformer(method='yeo-johnson')
y_full_pt = pt_full.fit_transform(y_full.reshape(-1, 1)).ravel()
pt_lo     = y_full_pt.min()
pt_hi     = y_full_pt.max()

# Gradient Boosting retrained on full data
gb_pt_full = GradientBoostingRegressor(
    n_estimators=2000, learning_rate=0.03, max_depth=7,
    subsample=0.9, min_samples_leaf=5,
    random_state=RANDOM_STATE
)
gb_pt_full.fit(X_base, y_full_pt)
test_gb_pt_raw = gb_pt_full.predict(X_test_base)
test_gb_pt_raw = np.clip(test_gb_pt_raw, pt_lo, pt_hi)
test_gb_pt     = np.maximum(pt_full.inverse_transform(
    test_gb_pt_raw.reshape(-1, 1)).ravel(), 0)
test_gb_pt     = np.nan_to_num(test_gb_pt, nan=0.0)

# LightGBM retrained on full data
lgb_full = lgb.LGBMRegressor(
    n_estimators=8000, learning_rate=0.01,
    max_depth=-1, num_leaves=63,
    subsample=0.9, colsample_bytree=0.9,
    min_child_samples=5,
    reg_alpha=0.05, reg_lambda=1.0,
    random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1
)
lgb_full.fit(X_base, y_full_pt)
test_lgb_raw = lgb_full.predict(X_test_base)
test_lgb_raw = np.clip(test_lgb_raw, pt_lo, pt_hi)
test_lgb     = np.maximum(pt_full.inverse_transform(
    test_lgb_raw.reshape(-1, 1)).ravel(), 0)
test_lgb     = np.nan_to_num(test_lgb, nan=0.0)

print('Test prediction ranges:')
print(f'  GB_PT: min={test_gb_pt.min():.3f}  max={test_gb_pt.max():.3f}  mean={test_gb_pt.mean():.3f}')
print(f'  LGB:   min={test_lgb.min():.3f}  max={test_lgb.max():.3f}  mean={test_lgb.mean():.3f}')
print(f'  NaNs — GB:{np.isnan(test_gb_pt).sum()}  LGB:{np.isnan(test_lgb).sum()}')

In [ ]:
# 4.14.4 Apply optimised weights to test predictions
test_mat2              = np.column_stack([test_gb_pt, test_lgb])
final_test_predictions = np.maximum(test_mat2 @ opt_weights, 0)

print('Final ensemble predictions:')
print(f'  min  = {final_test_predictions.min():.4f}')
print(f'  max  = {final_test_predictions.max():.4f}')
print(f'  mean = {final_test_predictions.mean():.4f}')

The weighted ensemble gives a better validation MAPE than any single model. The optimiser places most of the weight on LightGBM since it achieved the lowest individual MAPE, with Gradient Boosting contributing a smaller but meaningful share. Combining the two reduces variance compared to relying on either model alone.

---
## 5. Final Model and Kaggle Prediction

The final prediction uses the weighted ensemble from Section 4.14 - Gradient Boosting with PowerTransformer target at approximately 20% weight and LightGBM with PowerTransformer target at approximately 80% weight. Both models were retrained on the full training set to maximise information before generating test predictions. The ensemble achieved a Kaggle public leaderboard MAPE of **0.161348**.

In [ ]:
# 5.1 Save the final ensemble predictions to prediction.csv in the required format
prediction = sample_pred.copy()
prediction.iloc[:, 1] = final_test_predictions

prediction.to_csv('prediction.csv', index=False)

print('Prediction shape:', prediction.shape)
print(prediction.head())
print('Missing values:', prediction.isnull().sum().sum())
print('Minimum predicted pressure:', prediction.iloc[:, 1].min())
print('Maximum predicted pressure:', prediction.iloc[:, 1].max())
print('Mean predicted pressure:', prediction.iloc[:, 1].mean())

In [ ]:
# 5.2 Download the prediction file for Kaggle submission
from google.colab import files
files.download('prediction.csv')

The final Kaggle prediction uses a weighted ensemble of two models. For interpretation purposes a single Gradient Boosting model is fitted separately on the full training set, because feature importance, PDP and SHAP explanations are much cleaner on a single model than on a weighted ensemble. A log1p transformation is used for the target here rather than PowerTransformer because it guarantees positive predictions after applying expm1, which avoids any zero-floor issue in the local explanation plots.

In [ ]:
# 5.3 Fit a separate interpretable model on the full training set
# This model is used only for explanation, not for the Kaggle prediction
interpret_model = GradientBoostingRegressor(
    n_estimators=400, learning_rate=0.1, max_depth=6,
    random_state=RANDOM_STATE
)
interpret_model.fit(X_base, np.log1p(y_full))

X_interpret_full = X_base.copy()
y_interpret_full = y_full.copy()

# Generate training set predictions to use in the error analysis
y_pred_train = np.expm1(interpret_model.predict(X_interpret_full))

print('Interpretation model fitted.')
print('Feature set shape:', X_interpret_full.shape)
print('Min prediction:', y_pred_train.min())
print('Max prediction:', y_pred_train.max())

---
## 6. Model Interpretation

The interpretation section looks at what the model learned and why it makes the predictions it does. I use five methods: feature importance, permutation importance, partial dependence plots, SHAP, and local SHAP force plots for specific cases. All interpretation uses the dedicated Gradient Boosting model fitted in Section 5.

In [ ]:
# 6.0.1 Sample 1000 rows from the training set for SHAP and permutation importance
# Using all 10000 rows would be too slow for SHAP
np.random.seed(RANDOM_STATE)
n_interpret   = min(1000, len(X_interpret_full))
interpret_idx = np.random.choice(len(X_interpret_full), size=n_interpret, replace=False)

X_interpret = X_interpret_full.iloc[interpret_idx].reset_index(drop=True)
y_interpret = y_interpret_full[interpret_idx]

# 6.0.2 Generate predictions on the full training set for the error analysis
y_pred_train = np.expm1(interpret_model.predict(X_interpret_full))

# 6.0.3 Build error dataframe with actual, predicted and absolute percentage error
error_df = pd.DataFrame({
    'Actual':    y_interpret_full,
    'Predicted': y_pred_train,
    'APE':       np.abs((y_interpret_full - y_pred_train) / (y_interpret_full + 1e-9))
})

idx_lowest  = error_df['Predicted'].idxmin()
idx_highest = error_df['Predicted'].idxmax()
idx_worst   = error_df['APE'].idxmax()

print(f'Lowest prediction  : index {idx_lowest}   Predicted={error_df.loc[idx_lowest,  "Predicted"]:.4f}')
print(f'Highest prediction : index {idx_highest}  Predicted={error_df.loc[idx_highest, "Predicted"]:.4f}')
print(f'Largest APE        : index {idx_worst}    APE={error_df.loc[idx_worst, "APE"]:.4f}')

### 6.1 Feature Importance

The plot below shows which features the model relied on most, measured by total gain across all splits in the Gradient Boosting trees. All features are included, both original columns and the engineered ones created in Section 3.2.

In [ ]:
# 6.1.1 Compute feature importance from the interpretation model
importances        = pd.Series(interpret_model.feature_importances_,
                               index=X_interpret_full.columns)
importances_sorted = importances.sort_values(ascending=False)

# 6.1.2 Plot all features sorted by importance
fig, ax = plt.subplots(figsize=(10, max(6, len(importances_sorted) * 0.22)))
importances_sorted.plot(kind='barh', ax=ax)
ax.invert_yaxis()
ax.set_xlabel('Feature Importance (Gain)')
ax.set_title('Complete Feature Importance — All Features Including Engineered')
plt.tight_layout()
plt.show()

importances_sorted.head(15)

`Scaled Sensor Distance` and `Sensor ID` together make up about two thirds of total importance, which makes physical sense - how far the sensor is from the explosion and which wall it sits on both directly determine how much blast pressure it receives. `Obstacle Sensor Distance Difference` at 13% captures whether the sensor is in front of or behind the obstacle, which controls how much shielding it gets from the wall. Several of the engineered features appear in the top 15, including `Vapour Pressure Proxy`, `Superheat Volume`, and `Energy Proxy`, confirming that the physics-based feature engineering added genuine predictive value beyond the raw columns.

### 6.2 Permutation Importance

Permutation importance measures the drop in model performance when each feature is randomly shuffled. This is model-agnostic and does not depend on the tree structure, so it provides a useful independent check on the gain-based importance above.

In [ ]:
# 6.2.1 Compute permutation importance using interpret_model directly
perm_result = permutation_importance(
    interpret_model, X_interpret, np.log1p(y_interpret),
    scoring='neg_mean_absolute_percentage_error',
    n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1
)

perm_df = pd.DataFrame({
    'Feature':          X_interpret.columns,
    'Mean Importance':  perm_result.importances_mean,
    'Std':              perm_result.importances_std
}).sort_values('Mean Importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, max(6, len(perm_df) * 0.22)))
ax.barh(perm_df['Feature'], perm_df['Mean Importance'],
        xerr=perm_df['Std'], capsize=3)
ax.invert_yaxis()
ax.set_xlabel('Mean decrease in neg-MAPE')
ax.set_title('Permutation Feature Importance')
plt.tight_layout()
plt.show()

perm_df.head(15)

The permutation importance confirms the same story as the gain-based importance. Shuffling `Scaled Sensor Distance` or `Sensor ID` causes the largest performance drop at around 0.42 and 0.40 respectively, meaning the model is most dependent on knowing how far the sensor is from the blast and which wall it sits on. `Sensor Position Side` ranks third at 0.30 — which wall the sensor is on makes a big difference because the obstacle blocks and reflects the blast wave very differently on each side. Features near zero in the lower half of the chart add very little to the model's performance and could potentially be removed in a leaner version.

### 6.3 Partial Dependence Plots

PDPs show the average effect of changing one feature on the model's prediction, while holding all other features at their observed values. The five most important features from Section 6.1 are shown.

In [ ]:
# 6.3.1 Select top 5 features by gain importance for PDP
top5_features = importances_sorted.head(5).index.tolist()
print('Top 5 features for PDP:', top5_features)

# 6.3.2 Generate partial dependence plots using interpret_model directly
# The y-axis is in log-pressure scale (model output before expm1)
# Direction and shape of all relationships are correct
fig, ax = plt.subplots(figsize=(14, 8))
PartialDependenceDisplay.from_estimator(
    interpret_model, X_interpret,
    features=top5_features, grid_resolution=50, ax=ax
)
plt.suptitle('Partial Dependence Plots — Top 5 Features', y=1.02)
plt.tight_layout()
plt.show()

Looking at the plots from above:

**Scaled Sensor Distance** shows a sharp decrease from left to right — sensors close to the blast have very high predicted pressure, and as distance increases the pressure drops off quickly. The steepest part of the curve is at close range, which reflects the near-field amplification of blast waves.

**Sensor ID** shows a non-monotonic pattern with a peak around IDs 10 to 15 then a drop. This reflects the physical layout of sensors around the obstacle - certain positions on the front wall receive stronger direct loading while back-wall sensors receive less.

**Obstacle Sensor Distance Difference** shows a negative trend — as the sensor sits further behind the obstacle relative to the blast source, predicted pressure decreases. This captures the shielding effect of the wall.

**Sensor Position Side** shows three discrete levels corresponding to the three wall sides. Front-wall sensors have the highest predicted pressure, back-wall sensors the lowest.

**Sensor Position z** shows pressure increasing with height, which is consistent with sensors positioned closer to the BLEVE source height receiving more energy.

### 6.4 SHAP Summary Plot

SHAP values show how much each feature contributes to each individual prediction. The beeswarm plot below shows both the size and direction of each feature's effect across all 1000 sampled rows.

In [ ]:
# 6.4.1 Compute SHAP values using TreeExplainer on the 1000-row sample
explainer   = shap.TreeExplainer(interpret_model)
shap_values = explainer.shap_values(X_interpret)

# 6.4.2 Plot SHAP summary showing direction and magnitude for top 20 features
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_interpret, show=False, max_display=20)
plt.title('SHAP Summary Plot — Feature Effect on Predicted Pressure')
plt.tight_layout()
plt.show()

The beeswarm plot shows both which features matter most and in which direction they push predictions. Sensors with a low `Scaled Sensor Distance` (shown in red on the left side of the plot) strongly increase predicted pressure because they are close to the explosion. High distance values push predictions down. `Sensor ID` has a wide horizontal spread, meaning which wall the sensor is on has a large and variable effect depending on the specific location. `Obstacle Sensor Distance Difference` shows that sensors well behind the obstacle receive lower pressure due to shielding. The engineered features `Energy Proxy` and `Vapour Pressure Proxy` appear in the top contributors, which confirms that the physics-based feature engineering added real value.

### 6.5 SHAP Feature Interaction Plots

SHAP dependence plots show how the effect of one feature changes depending on the value of another. Two pairs of features with meaningful physical interactions are shown below.

In [ ]:
# 6.5.1 SHAP dependence plot for the top two features
# Colour shows the value of the interaction feature
top_features = importances_sorted.head(5).index.tolist()
print('Features used for dependence plots:', top_features[:4])

# Plot 1: Scaled Sensor Distance coloured by Sensor ID
shap.dependence_plot(
    top_features[0], shap_values, X_interpret,
    interaction_index=top_features[1]
)

# Plot 2: Obstacle Sensor Distance Difference coloured by Sensor Position Side
shap.dependence_plot(
    top_features[2], shap_values, X_interpret,
    interaction_index=top_features[3]
)

**Plot 1 - Scaled Sensor Distance coloured by Sensor ID:** At small distances the SHAP contribution is strongly positive regardless of which sensor, meaning proximity to the blast dominates everything else. As distance increases, the Sensor ID colour starts to separate — front-wall sensors keep higher pressure contributions at the same distance compared to back-wall sensors. This shows that distance and wall position interact, and that being on the front wall amplifies the pressure at any given distance.

**Plot 2 - Obstacle Sensor Distance Difference coloured by Sensor Position Side:** Sensors on the front wall show positive SHAP contributions even at moderate distances, while back-wall sensors turn negative as they move further behind the obstacle. This directly captures the shielding interaction - the obstacle blocks energy for back-wall sensors but front-wall sensors are still exposed to direct blast loading.

### 6.6 Local Explanations - SHAP Force Plots

Three specific training examples are explained individually using SHAP force plots. The three cases are the lowest prediction, the highest prediction, and the case with the largest absolute percentage error.

In [ ]:
# 6.6.1 Compute SHAP values for the three key cases
key_indices = [idx_lowest, idx_highest, idx_worst]
key_labels  = ['Lowest Prediction', 'Highest Prediction', 'Largest APE']

key_X    = X_interpret_full.iloc[key_indices]
shap_key = explainer.shap_values(key_X)
base_val = explainer.expected_value

shap.initjs()
print('SHAP values computed for the three key cases.')

In [ ]:
# 6.6.2 Force plot for the lowest prediction
print(f'=== {key_labels[0]} ===')
print(error_df.loc[idx_lowest])
shap.force_plot(base_val, shap_key[0], key_X.iloc[0],
                matplotlib=True, show=False, figsize=(16, 3))
plt.title('SHAP Force Plot — Lowest Prediction')
plt.tight_layout()
plt.show()

The model predicted 0.0110 bar for this sensor while the actual was 0.0161 bar. This sensor is far from the explosion and positioned behind the obstacle on the back wall, which is why the distance and shielding features push the prediction well below the baseline. The model is in the right ballpark - the prediction is lower than actual but in the same order of magnitude.

In [ ]:
# 6.6.3 Force plot for the highest prediction
print(f'=== {key_labels[1]} ===')
print(error_df.loc[idx_highest])
shap.force_plot(base_val, shap_key[1], key_X.iloc[1],
                matplotlib=True, show=False, figsize=(16, 3))
plt.title('SHAP Force Plot — Highest Prediction')
plt.tight_layout()
plt.show()

The model predicted 9.088 bar against an actual of 9.170 bar, which is nearly perfect with an APE of only 0.009. This sensor is very close to the explosion on the front wall. The force plot shows multiple large positive contributions from the distance and energy features all stacking together, reflecting a sensor directly in the path of the full blast wave with nothing blocking it.

In [ ]:
# 6.6.4 Force plot for the largest absolute percentage error
print(f'=== {key_labels[2]} ===')
print(error_df.loc[idx_worst])
shap.force_plot(base_val, shap_key[2], key_X.iloc[2],
                matplotlib=True, show=False, figsize=(16, 3))
plt.title('SHAP Force Plot — Largest Absolute Percentage Error')
plt.tight_layout()
plt.show()

The model predicted 0.0437 bar but the actual was only 0.0229 bar, giving an APE of 0.91. The model overestimated by roughly double. The force plot shows conflicting contributions pulling in opposite directions, which suggests this sensor is in an unusual geometry that is underrepresented in the training set. The overestimation is also partly a consequence of MAPE being the evaluation metric — for very low pressure values, even a small absolute error creates a large relative error, so these edge cases are inherently difficult to get right.

## 7. Conclusion

This notebook built a machine learning pipeline to predict BLEVE peak pressure around a rigid obstacle, going from raw data exploration all the way to a final Kaggle submission.

**What I found:**

Linear models completely failed on this problem with MAPE above 1.0, which confirmed early on that the pressure data is highly nonlinear and a simple approach would not work. Tree-based ensemble models were the only approach that produced useful predictions.

The physics-informed feature engineering made a genuine difference. `Scaled Sensor Distance`, `Energy Proxy` and `Pressure Volume` appeared consistently in the top features across all three importance methods — gain-based importance, permutation importance and SHAP — which confirmed they were not just noise but actually encoding real blast physics that the model could use.

Applying a PowerTransformer to the target variable improved MAPE for both Gradient Boosting and LightGBM by normalising the right-skewed pressure distribution before training. This came with a trade-off where R² dropped because clipping before inverse transformation compressed the high-pressure tail. The non-transformed Gradient Boosting model at R²=0.948 shows the models were genuinely learning the data structure, and the tuned PowerTransformer GB model managed to achieve both R²=0.88 and MAPE=0.1054 simultaneously after systematic hyperparameter tuning beyond what GridSearchCV alone could find.

The systematic fine-tuning of `max_depth`, `subsample` and `min_samples_leaf` was the single biggest improvement in the project. Moving from the GridSearchCV baseline (MAPE=0.1492) to the fully tuned model (MAPE=0.1054) was a much larger gain than any ensemble combination, which shows that getting the individual model right matters more than how you combine models.

The final weighted ensemble of tuned Gradient Boosting (approximately 20% weight) and LightGBM (approximately 80% weight) achieved a Kaggle public leaderboard MAPE of **0.161348**, well below the baseline of 0.252786.

SHAP analysis confirmed that `Scaled Sensor Distance`, `Sensor ID` and `Obstacle Sensor Distance Difference` are the three most influential features. The PDP and dependence plots both showed physically coherent behaviour - pressure decaying sharply with distance, front-wall sensors experiencing higher loading than back-wall sensors, and the obstacle providing meaningful shielding for sensors positioned behind it.

**Limitations:**

The case with the largest APE of 0.91 involved a low-pressure sensor in an unusual geometry where the model overestimated by roughly double. Very low pressure cases are inherently difficult under MAPE because even a small absolute error produces a large relative one, so these edge cases will always be a challenge for this metric.

There was a gap of about 0.04 to 0.05 MAPE points between validation and Kaggle performance throughout the project. Some of this was due to overfitting the ensemble weights to the specific validation split, and some was likely genuine distribution differences between training and test cases.

XGBoost and Gradient Boosting trained on sensor dummy-coded features consistently produced compressed test predictions with a maximum of around 0.33 to 0.79 bar instead of the expected 9.17 bar. This appears to be a distribution mismatch between the sensor dummy columns in training and test, and it meant sensor-coded variants had to be excluded from the final ensemble despite showing strong validation performance.

Getting below the 0.15 MAPE threshold that some other students achieved was not possible despite LightGBM being the strongest individual model tested. Further improvement would likely require additional data or a different modelling approach entirely.